In [0]:
#— Configuration

CATALOG = "adb_retailedge_dev"
SCHEMA  = "healthcare_cms"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")

Catalog : adb_retailedge_dev
Schema  : healthcare_cms


In [0]:
# Silver Providers (Clean + Select Key Columns)
from pyspark.sql.functions import col, trim, expr

df_providers = spark.table(f"{CATALOG}.{SCHEMA}.bronze_providers")

silver_providers = df_providers \
    .select(
      trim(col("Rndrng_NPI")).alias("provider_npi"),
      trim(col("Rndrng_Prvdr_Last_Org_Name")).alias("provider_name"),
      trim(col("Rndrng_Prvdr_First_Name")).alias("provider_first_name"),
      trim(col("Rndrng_Prvdr_City")).alias("city"),
      trim(col("Rndrng_Prvdr_State_Abrvtn")).alias("state"),
      trim(col("Rndrng_Prvdr_Zip5")).alias("zip_code"),
      trim(col("Rndrng_Prvdr_Type")).alias("provider_type"),
      trim(col("HCPCS_Cd")).alias("hcpcs_code"),
      trim(col("HCPCS_Desc")).alias("hcpcs_description"),
      trim(col("Place_Of_Srvc")).alias("place_of_service"),
      expr("try_cast(Tot_Benes as int)").alias("total_beneficiaries"),
      expr("try_cast(Tot_Srvcs as double)").alias("total_services"),
      expr("try_cast(Avg_Mdcr_Pymt_Amt as double)").alias("avg_medicare_payment"),
      expr("try_cast(Avg_Sbmtd_Chrg as double)").alias("avg_submitted_charge"),
      expr("try_cast(Avg_Mdcr_Alowd_Amt as double)").alias("avg_allowed_amount")
    ) \
    .filter(col("provider_npi").isNotNull()) \
    .filter(col("state").isNotNull())

silver_providers.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_providers")

count = spark.table(f"{CATALOG}.{SCHEMA}.silver_providers").count()
print(f"silver_providers: {count:,} rows written")

silver_providers: 592,965 rows written


In [0]:
# Silver Provider Summery
from pyspark.sql.functions import sum, avg, count, round

silver_provider_summary = spark.table(f"{CATALOG}.{SCHEMA}.silver_providers") \
    .groupBy("provider_npi","provider_name","provider_first_name","city","state","zip_code","provider_type") \
    .agg(
        count("hcpcs_code").alias("total_procedures"),
        sum("total_beneficiaries").alias("total_beneficiaries"),
        sum("total_services").alias("total_services"),
        round(avg("avg_medicare_payment"),2).alias("avg_medicare_payment"),
        round(avg("avg_submitted_charge"),2).alias("avg_submitted_charge"),
        round(avg("avg_allowed_amount"),2).alias("avg_allowed_amount")
    )

silver_provider_summary.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_provider_summary")

count = spark.table(f"{CATALOG}.{SCHEMA}.silver_provider_summary").count()
print(f"silver_provider_summary: {count:,} rows written")



silver_provider_summery: 73,673 rows written


In [0]:
# Silver Drugs
from pyspark.sql.functions import col, trim, expr

silver_drugs = spark.table(f"{CATALOG}.{SCHEMA}.bronze_drugs")\
    .select(
        trim(col("Brnd_Name")).alias("brand_name"),
        trim(col("Gnrc_Name")).alias("generic_name"),
        trim(col("Mftr_Name")).alias("manufacturer"),
        expr("try_cast(Tot_Mftr as int)").alias("total_manufacturers"),
        expr("try_cast(Tot_Clms_2024 as int)").alias("total_claims_2024"),
        expr("try_cast(Tot_Benes_2024 as int)").alias("total_beneficiaries_2024"),
        expr("try_cast(Tot_Spndng_2024 as double)").alias("total_spending_2024"),
        expr("try_cast(Avg_Spnd_Per_Clm_2024 as double)").alias("avg_spend_per_claim_2024"),
        expr("try_cast(Avg_Spnd_Per_Bene_2024 as double)").alias("avg_spend_per_bene_2024"),
        expr("try_cast(Tot_Spndng_2023 as double)").alias("total_spending_2023"),
        expr("try_cast(Tot_Spndng_2022 as double)").alias("total_spending_2022"),
        expr("try_cast(Tot_Spndng_2021 as double)").alias("total_spending_2021"),
        expr("try_cast(Tot_Spndng_2020 as double)").alias("total_spending_2020"),
        expr("try_cast(CAGR_Avg_Spnd_Per_Dsg_Unt_20_24 as double)").alias("cagr_2020_2024"),
        col("Outlier_Flag_2024").alias("outlier_flag")
      ) \
      .filter(col("brand_name").isNotNull()) \
      .filter(col("total_spending_2024").isNotNull())
        
silver_drugs.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_drugs")

count = spark.table(f"{CATALOG}.{SCHEMA}.silver_drugs").count()
print(f"silver_drugs: {count:,} rows written")

silver_drugs: 14,536 rows written


In [0]:
# Silver Inpatient

from pyspark.sql.functions import col, trim, expr

silver_inpatient = spark.table(f"{CATALOG}.{SCHEMA}.bronze_inpatient") \
    .select(
        trim(col("Rndrng_Prvdr_Geo_Lvl")).alias("geo_level"),
        trim(col("Rndrng_Prvdr_Geo_Cd")).alias("geo_code"),
        trim(col("Rndrng_Prvdr_Geo_Desc")).alias("geo_description"),
        trim(col("DRG_Cd")).alias("drg_code"),
        trim(col("DRG_Desc")).alias("drg_description"),
        expr("try_cast(Tot_Dschrgs as int)").alias("total_discharges"),
        expr("try_cast(Avg_Submtd_Cvrd_Chrg as double)").alias("avg_submitted_charge"),
        expr("try_cast(Avg_Tot_Pymt_Amt as double)").alias("avg_total_payment"),
        expr("try_cast(Avg_Mdcr_Pymt_Amt as double)").alias("avg_medicare_payment")
    ) \
    .filter(col("geo_level") == "State") \
    .filter(col("drg_code").isNotNull()) \
    .filter(col("total_discharges").isNotNull())

silver_inpatient.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_inpatient")

count = spark.table(f"{CATALOG}.{SCHEMA}.silver_inpatient").count()
print(f"silver_inpatient: {count:,} rows written")

silver_inpatient: 25,804 rows written


In [0]:
# Silver Verification

tables = ["providers", "provider_summary", "drugs", "inpatient"]
total = 0

print("=" * 50)
print("SILVER LAYER VERIFICATION")
print("=" * 50)
for t in tables:
    count = spark.table(f"{CATALOG}.{SCHEMA}.silver_{t}").count()
    total += count
    print(f"silver_{t:<20}: {count:>10,} rows")

print("-" * 50)
print(f"{'TOTAL':<25}: {total:>10,} rows")
print("=" * 50)

SILVER LAYER VERIFICATION
silver_providers           :    592,965 rows
silver_provider_summary    :     73,673 rows
silver_drugs               :     14,536 rows
silver_inpatient           :     25,804 rows
--------------------------------------------------
TOTAL                    :    706,978 rows


Table renamed successfully
